In [1]:
from collections import Counter
from Bio import SeqIO
import matplotlib.pyplot as plt
import random
import pandas as pd

In [2]:
used_classes = {
    'Actinopteri_186623',
    'Aves_8782',
    'Mammalia_40674',
    'Lepidosauria_8504',
    'Amphibia_8292',
    'Chondrichthyes_7777',
}

In [3]:
cytb_longest = SeqIO.parse('./MIDORI2_LONGEST_NUC_GB264_Cytb_BLAST.fasta', 'fasta')
human_size = 1140
shift = 5
sizes = []
classes = []
verts = []
for rec in cytb_longest:
    if 'Chordata_7711' in rec.name:
        classes.append(rec.name.split(';')[3])
        sizes.append(len(rec))
        if human_size-shift < len(rec) < human_size+shift and rec.name.split(';')[3] in used_classes:
            verts.append(rec)
plt.hist([x for x in sizes if x > 1000], bins=100)
plt.vlines([human_size-shift, human_size+shift], 0, 10000, 'r', '--')
random.shuffle(verts)

In [4]:
Counter(classes)

Counter({'Actinopteri_186623': 13355,
         'Aves_8782': 7282,
         'Mammalia_40674': 5199,
         'Lepidosauria_8504': 4615,
         'Amphibia_8292': 3265,
         'Chondrichthyes_7777': 407,
         'class_Testudines_8459': 355,
         'Ascidiacea_7713': 52,
         'Hyperoartia_117569': 45,
         'class_Crocodylia_1294634': 27,
         'Cladistia_1338366': 12,
         'Leptocardii_2682552': 8,
         'Thaliacea_30304': 7,
         'Myxini_117565': 6,
         'class_Ceratodontiformes_2823314': 4,
         'class_Coelacanthiformes_7894': 2})

In [5]:
len(sizes), len(verts)

(34641, 16078)

In [6]:
# derive outgrp
cytb_longest = SeqIO.parse('./MIDORI2_LONGEST_NUC_GB264_Cytb_BLAST.fasta', 'fasta')
human_size = 1140
shift = 30
outgrp = []
sizes = []
for rec in cytb_longest:
    if 'Ascidiacea_7713' in rec.name and len(rec) > 1000:
        sizes.append(len(rec))
        if human_size-shift < len(rec) < human_size+shift and rec.name.split(';')[3]:
            outgrp.append(rec)
            break
plt.hist(sizes)
plt.title('Potential OUTGRP seq sizes')
outgrp[0].id = 'OUTGRP|' + outgrp[0].id
outgrp

[SeqRecord(seq=Seq('ATTGTTATGTTTCGTAAAAATGAACTTGTTAGTCTTTTTTTAAGATCATTTAGT...GAT'), id='OUTGRP|HG931923.1.7035.8148###root_1;Eukaryota_2759;Chordata_7711;Ascidiacea_7713;Stolidobranchia_7720;Styelidae_7721;Botryllus_30300;Botryllus_schlosseri_30301', name='HG931923.1.7035.8148###root_1;Eukaryota_2759;Chordata_7711;Ascidiacea_7713;Stolidobranchia_7720;Styelidae_7721;Botryllus_30300;Botryllus_schlosseri_30301', description='HG931923.1.7035.8148###root_1;Eukaryota_2759;Chordata_7711;Ascidiacea_7713;Stolidobranchia_7720;Styelidae_7721;Botryllus_30300;Botryllus_schlosseri_30301', dbxrefs=[])]

In [7]:
SeqIO.write(outgrp + verts[:5000], 'dataset2/chordates5k.fa', 'fasta-2line')

5001

In [8]:
Counter(outgrp[0].seq)

Counter({'T': 517, 'A': 279, 'G': 176, 'C': 142})

## taxa distribution approach

In [9]:
taxa = pd.DataFrame([x.name.split(';') for x in verts]).iloc[:, 3:]
taxa

,3,4,5,6,7
0,Actinopteri_186623,Siluriformes_7995,Ictaluridae_7996,Noturus_61322,Noturus_nocturnus_90667
1,Actinopteri_186623,Cypriniformes_7952,Leuciscidae_2743726,Richardsonius_86929,Richardsonius_balteatus_86930
2,Aves_8782,Passeriformes_9126,Pittidae_9161,Pitta_9162,Pitta_nympha_397308
3,Mammalia_40674,Chiroptera_9397,Phyllostomidae_9415,Platyrrhinus_27657,Platyrrhinus_helleri_27658
4,Lepidosauria_8504,Squamata_8509,Scincidae_66056,Ctenotus_204894,Ctenotus_spaldingi_480784
...,...,...,...,...,...
16073,Mammalia_40674,Rodentia_9989,Heteromyidae_10015,Dipodomys_10016,Dipodomys_venustus_323375
16074,Aves_8782,Procellariiformes_30449,Diomedeidae_37067,Phoebastria_521259,Phoebastria_nigripes_54021
16075,Actinopteri_186623,Perciformes_8111,Cottidae_8092,Cottus_8098,Cottus_marginatus_304283
16076,Actinopteri_186623,Siluriformes_7995,Callichthyidae_31012,Brochis_245762,Brochis_multiradiatus_337654


### prepare one species from genus

In [10]:
name_size_cutoff = 28

genus_df = taxa[6].value_counts().reset_index()
genus_df.columns = ['genus', 'nspecies']
genus_df['name_size'] = genus_df.genus.str.len()

genus_df.name_size.hist(bins=50)
plt.vlines(name_size_cutoff, 0, 1200, 'r', '--')
plt.show()
genus_df = genus_df[genus_df.name_size < name_size_cutoff]
genus_df.sort_values('name_size')

/tmp/ipykernel_86056/930300688.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,genus,nspecies,name_size
474,Sus_9822,7,8
3071,Aix_8831,1,8
296,Ara_9225,9,8
291,Bos_9903,10,8
1888,Pan_9596,2,8
...,...,...,...
791,Lepidocephalichthys_425478,4,26
3716,Amblychaeturichthys_497221,1,26
1202,Pseudoscaphirhynchus_55289,3,26
3235,Tenebrosternarchus_2936481,1,26


In [11]:
genus_df.genus.str.count('_').value_counts()

genus
1    5046
Name: count, dtype: int64

In [12]:
taxa

,3,4,5,6,7
0,Actinopteri_186623,Siluriformes_7995,Ictaluridae_7996,Noturus_61322,Noturus_nocturnus_90667
1,Actinopteri_186623,Cypriniformes_7952,Leuciscidae_2743726,Richardsonius_86929,Richardsonius_balteatus_86930
2,Aves_8782,Passeriformes_9126,Pittidae_9161,Pitta_9162,Pitta_nympha_397308
3,Mammalia_40674,Chiroptera_9397,Phyllostomidae_9415,Platyrrhinus_27657,Platyrrhinus_helleri_27658
4,Lepidosauria_8504,Squamata_8509,Scincidae_66056,Ctenotus_204894,Ctenotus_spaldingi_480784
...,...,...,...,...,...
16073,Mammalia_40674,Rodentia_9989,Heteromyidae_10015,Dipodomys_10016,Dipodomys_venustus_323375
16074,Aves_8782,Procellariiformes_30449,Diomedeidae_37067,Phoebastria_521259,Phoebastria_nigripes_54021
16075,Actinopteri_186623,Perciformes_8111,Cottidae_8092,Cottus_8098,Cottus_marginatus_304283
16076,Actinopteri_186623,Siluriformes_7995,Callichthyidae_31012,Brochis_245762,Brochis_multiradiatus_337654


In [13]:
taxa_species = taxa.drop_duplicates(subset=6)
taxa_species = taxa_species[taxa_species[6].isin(genus_df.genus)]
taxa_species

,3,4,5,6,7
0,Actinopteri_186623,Siluriformes_7995,Ictaluridae_7996,Noturus_61322,Noturus_nocturnus_90667
1,Actinopteri_186623,Cypriniformes_7952,Leuciscidae_2743726,Richardsonius_86929,Richardsonius_balteatus_86930
2,Aves_8782,Passeriformes_9126,Pittidae_9161,Pitta_9162,Pitta_nympha_397308
3,Mammalia_40674,Chiroptera_9397,Phyllostomidae_9415,Platyrrhinus_27657,Platyrrhinus_helleri_27658
4,Lepidosauria_8504,Squamata_8509,Scincidae_66056,Ctenotus_204894,Ctenotus_spaldingi_480784
...,...,...,...,...,...
16040,Actinopteri_186623,Cypriniformes_7952,Xenocyprididae_2743745,Araiocypris_3113019,Araiocypris_batodes_3113020
16045,Mammalia_40674,Rodentia_9989,Muridae_10066,Oenomys_332667,Oenomys_hypoxanthus_332668
16046,Actinopteri_186623,Labriformes_1489928,Odacidae_241355,Olisthops_1605465,Olisthops_cyanomelas_1605466
16050,Actinopteri_186623,Gerreiformes_2024538,Gerreidae_274463,Diapterus_334384,Diapterus_auratus_334385


In [14]:
# some species have no family name
taxa_species[taxa_species[5].str.count('_') > 1]

,3,4,5,6,7
19,Aves_8782,Passeriformes_9126,family_Pachycephala_46903,Pachycephala_46903,Pachycephala_cinerea_449359
962,Aves_8782,Passeriformes_9126,family_Rhagologus_156169,Rhagologus_156169,Rhagologus_leucostigma_156170
2096,Aves_8782,Procellariiformes_30449,family_Garrodia_79648,Garrodia_79648,Garrodia_nereis_79649
2189,Aves_8782,Passeriformes_9126,family_Oreoica_254565,Oreoica_254565,Oreoica_gutturalis_254566
2368,Aves_8782,Passeriformes_9126,family_Colluricincla_156159,Colluricincla_156159,Colluricincla_harmonica_254449
2695,Aves_8782,Passeriformes_9126,family_Daphoenositta_254527,Daphoenositta_254527,Daphoenositta_chrysoptera_254528
3338,Aves_8782,Passeriformes_9126,family_Myzornis_634317,Myzornis_634317,Myzornis_pyrrhoura_634318
3618,Aves_8782,Passeriformes_9126,family_Platysteira_107225,Platysteira_107225,Platysteira_castanea_1160851
3690,Actinopteri_186623,Perciformes_8111,family_Bembrops_270608,Bembrops_270608,Bembrops_greyi_1003164
3980,Aves_8782,Procellariiformes_30449,family_Fregetta_37072,Fregetta_37072,Fregetta_maoriana_1112029


In [15]:
taxa_species.groupby(5)[7].count().hist(bins=40)

<Axes: title={'center': 'Potential OUTGRP seq sizes'}>

In [16]:
selected_species = set(taxa_species[7])
chord_sample = [x for x in verts if x.name.split(';')[-1] in selected_species]

SeqIO.write(outgrp + chord_sample, 'dataset2/chordates_sample.fa', 'fasta-2line')

5047

In [17]:
# next this fasta file was aligned as amino acids and manually filtered
# now we load saved seq ids and again prepare nucleotide fasta
with open('./dataset2/clear_seq_ids.txt') as fin:
    saved_ids = [x.strip().split('.')[0] for x in fin.readlines()]
saved_ids[:5], len(saved_ids)

(['OUTGRP|HG931923', 'LC564971', 'MN885507', 'AY208568', 'AY926394'], 4992)

In [18]:
from copy import copy

selected_species = set(saved_ids)
chord_sample = [x for x in verts if x.name.split('.')[0] in selected_species]
chord_sample_renamed = []
for x in chord_sample:
    x = copy(x)
    x.id = x.name.split(';')[-1]
    x.name = x.name.split(';')[-1]
    x.description = x.name.split(';')[-1]
    chord_sample_renamed.append(x)

outgrp_renamed = copy(outgrp[0])
outgrp_renamed.id = 'OUTGRP ' + outgrp_renamed.name.split(';')[-1]
outgrp_renamed.name = ''
outgrp_renamed.description = ''


SeqIO.write([outgrp_renamed] + chord_sample_renamed, 
            'dataset2/chordates_sample_v2.fa', 'fasta-2line')

4992

In [19]:
outgrp_renamed.name

''

In [20]:
outgrp[0].name

'HG931923.1.7035.8148###root_1;Eukaryota_2759;Chordata_7711;Ascidiacea_7713;Stolidobranchia_7720;Styelidae_7721;Botryllus_30300;Botryllus_schlosseri_30301'